# LoL Draft Win Rate Predictor v2

**Pipeline:**
1. Data Wrangling — role-specific win rates & synergy, no bans
2. Feature Engineering — champion_role entities with fallback threshold
3. Model Training — LightGBM + isotonic calibration
4. Evaluation — Brier Score, Log-Loss, calibration curve
5. SHAP Analysis — global + per-match explainability
6. Save Artifacts — model, encoders, lookup maps
7. Inference Function — live partial-draft prediction for the frontend

**Key changes from v1:**
- Bans removed from pipeline (model never used them)
- Champions treated as role-specific entities: `Ashe_bot` ≠ `Ashe_sup`
- Threshold fallback: if `champion_role` has < 20 games, use global champion win rate
- `picks_filled` feature added to help calibrate partial-draft predictions
- All feature engineering extracted into reusable `build_features()` function

In [ ]:
# ── 0. Install dependencies ──────────────────────────────────────────────────
!pip install polars lightgbm shap scikit-learn matplotlib pandas numpy --quiet

In [ ]:
# ── 1. Imports & Drive mount ─────────────────────────────────────────────────
import os, json, pickle
from itertools import combinations

import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt

from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss, log_loss
from sklearn.preprocessing import LabelEncoder

from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/Esports_project"
PARQUET_DIR = f"{BASE}/Parquets_v2"
os.makedirs(PARQUET_DIR, exist_ok=True)
print("Base:", BASE)
print("Parquet dir:", PARQUET_DIR)

---
## Section 1 — Data Wrangling

In [ ]:
# ── 1.1 Load raw CSV ─────────────────────────────────────────────────────────
CSV_PATH = f"{BASE}/2026_LoL_esports_match_data_from_OraclesElixir.csv"
df_raw = pl.read_csv(CSV_PATH, infer_schema_length=10000)

print("Shape:", df_raw.shape)
print("Positions:", df_raw["position"].value_counts())

In [ ]:
# ── 1.2 Split into player rows and team rows ─────────────────────────────────
# Each game = 10 player rows (5 per team) + 2 team summary rows
players = df_raw.filter(pl.col("position") != "team")
teams   = df_raw.filter(pl.col("position") == "team")

print("Player rows:", players.shape)
print("Team rows  :", teams.shape)

In [ ]:
# ── 1.3 Global champion win rates (fallback) ─────────────────────────────────
# Used when a champion_role combo has too few games to be reliable
GLOBAL_WR_MIN_GAMES = 10

champ_wr_global = (
    players
    .group_by("champion")
    .agg([
        pl.col("result").mean().alias("win_rate"),
        pl.len().alias("games")
    ])
    .filter(pl.col("games") >= GLOBAL_WR_MIN_GAMES)
    .sort("win_rate", descending=True)
)

global_avg_wr = float(champ_wr_global["win_rate"].mean())
print(f"Global avg win rate: {global_avg_wr:.4f}")
print(champ_wr_global.head(10))

In [ ]:
# ── 1.4 Role-specific champion win rates (primary) ───────────────────────────
# Key insight: Ashe played as support has a completely different win rate
# than Ashe played as ADC. We treat them as separate entities.
#
# Threshold: >= 20 games required for a champion_role to be used.
# Below threshold -> fall back to global champion win rate.

ROLE_WR_MIN_GAMES = 20

champ_role_wr = (
    players
    .with_columns(
        (pl.col("champion") + "_" + pl.col("position")).alias("champion_role")
    )
    .group_by("champion_role")
    .agg([
        pl.col("result").mean().alias("win_rate"),
        pl.len().alias("games")
    ])
    .filter(pl.col("games") >= ROLE_WR_MIN_GAMES)
    .sort("win_rate", descending=True)
)

print(f"Unique champion_role combos (>={ROLE_WR_MIN_GAMES} games): {len(champ_role_wr)}")
print(champ_role_wr.head(15))

In [ ]:
# ── 1.5 Role-specific synergy pairs ─────────────────────────────────────────
# Pair key: ("Jinx_bot", "Thresh_sup") — role-aware synergy
# Threshold: >= 5 games together

PAIR_MIN_GAMES = 5

players_with_role = players.with_columns(
    (pl.col("champion") + "_" + pl.col("position")).alias("champion_role")
)

pairs = (
    players_with_role.select(["gameid", "teamname", "champion_role", "result"])
    .join(
        players_with_role.select(["gameid", "teamname", "champion_role"])
                         .rename({"champion_role": "champ_role2"}),
        on=["gameid", "teamname"]
    )
    .filter(pl.col("champion_role") < pl.col("champ_role2"))
    .group_by(["champion_role", "champ_role2"])
    .agg([
        pl.col("result").mean().alias("pair_win_rate"),
        pl.len().alias("games_together")
    ])
    .filter(pl.col("games_together") >= PAIR_MIN_GAMES)
    .sort("pair_win_rate", descending=True)
)

print(f"Unique role-specific pairs: {len(pairs)}")
print(pairs.head(15))

In [ ]:
# ── 1.6 Build final_draft — no bans ─────────────────────────────────────────
# Only keep what the model actually uses: picks, game context, result.

draft_base = players.select([
    "gameid", "league", "year", "split",
    "playoffs", "date", "patch",
    "side", "position", "teamname",
    "firstPick", "champion",
    "result"
])

# Pivot: one row per game, columns = blue_top, blue_jng, ..., red_sup
draft_with_role = draft_base.with_columns(
    (
        pl.col("side").str.to_lowercase()
        + "_"
        + pl.col("position")
    ).alias("side_position")
)

champion_pivot = (
    draft_with_role
    .select(["gameid", "side_position", "champion"])
    .pivot(
        values="champion",
        index="gameid",
        on="side_position",
        aggregate_function="first"
    )
)

# Game context (one row per game)
game_context = (
    draft_base
    .group_by("gameid")
    .agg([
        pl.col("league").first(),
        pl.col("year").first(),
        pl.col("split").first(),
        pl.col("playoffs").first(),
        pl.col("date").first(),
        pl.col("patch").first()
    ])
)

# Team names + first pick side (no bans)
side_info = (
    draft_base
    .group_by(["gameid", "side"])
    .agg([
        pl.col("teamname").first(),
        pl.col("firstPick").first()
    ])
)

blue_info = (
    side_info.filter(pl.col("side") == "Blue")
    .select([
        "gameid",
        pl.col("teamname").alias("blue_team"),
        pl.col("firstPick").alias("blue_firstPick")
    ])
)

red_info = (
    side_info.filter(pl.col("side") == "Red")
    .select([
        "gameid",
        pl.col("teamname").alias("red_team"),
        pl.col("firstPick").alias("red_firstPick")
    ])
)

# Blue side result as the target label (1 = blue wins, 0 = red wins)
blue_results = (
    teams.filter(pl.col("side") == "Blue")
    .select(["gameid", pl.col("result").alias("blue_win")])
)

# Join everything
final_draft = (
    champion_pivot
    .join(game_context, on="gameid", how="left")
    .join(blue_info,    on="gameid", how="left")
    .join(red_info,     on="gameid", how="left")
    .join(blue_results, on="gameid", how="left")
)

print("final_draft shape:", final_draft.shape)
print("Columns:", final_draft.columns)
print("Null counts:")
print(final_draft.null_count())
print("\nTarget distribution:")
print(final_draft["blue_win"].value_counts())

In [ ]:
# ── 1.7 Save parquets ────────────────────────────────────────────────────────
final_draft.write_parquet(f"{PARQUET_DIR}/final_draft.parquet")
champ_wr_global.write_parquet(f"{PARQUET_DIR}/champ_winrates_global.parquet")
champ_role_wr.write_parquet(f"{PARQUET_DIR}/champ_winrates_role.parquet")
pairs.write_parquet(f"{PARQUET_DIR}/champ_synergies.parquet")
players.write_parquet(f"{PARQUET_DIR}/players.parquet")
teams.write_parquet(f"{PARQUET_DIR}/teams.parquet")

print("Parquets saved to:", PARQUET_DIR)

---
## Section 2 — Feature Engineering

In [ ]:
# ── 2.1 Load parquets (start here if resuming) ───────────────────────────────
final_draft  = pl.read_parquet(f"{PARQUET_DIR}/final_draft.parquet")
champ_wr_global = pl.read_parquet(f"{PARQUET_DIR}/champ_winrates_global.parquet")
champ_role_wr   = pl.read_parquet(f"{PARQUET_DIR}/champ_winrates_role.parquet")
pairs           = pl.read_parquet(f"{PARQUET_DIR}/champ_synergies.parquet")

# Build lookup maps (used by build_features and inference)
champ_role_wr_map = dict(zip(
    champ_role_wr["champion_role"].to_list(),
    champ_role_wr["win_rate"].to_list()
))
champ_wr_global_map = dict(zip(
    champ_wr_global["champion"].to_list(),
    champ_wr_global["win_rate"].to_list()
))
global_avg_wr = float(champ_wr_global["win_rate"].mean())

pair_map = {}
for row in pairs.to_dicts():
    key = tuple(sorted([row["champion_role"], row["champ_role2"]]))
    pair_map[key] = row["pair_win_rate"]

# Role suffix lookup
ROLE_COLS = ["blue_top", "blue_jng", "blue_mid", "blue_bot", "blue_sup",
             "red_top",  "red_jng",  "red_mid",  "red_bot",  "red_sup"]
COL_TO_ROLE = {c: c.split("_")[1] for c in ROLE_COLS}
BLUE_ROLES = [c for c in ROLE_COLS if c.startswith("blue")]
RED_ROLES  = [c for c in ROLE_COLS if c.startswith("red")]

print(f"Role WR map entries : {len(champ_role_wr_map)}")
print(f"Global WR map entries: {len(champ_wr_global_map)}")
print(f"Pair map entries    : {len(pair_map)}")
print(f"Global avg WR       : {global_avg_wr:.4f}")

In [ ]:
# ── 2.2 Win rate lookup with role-specific fallback ──────────────────────────
def get_wr(champion, role):
    """
    Priority:
      1. Role-specific win rate (e.g. Ashe_bot)
      2. Global champion win rate (Ashe, any role)
      3. Dataset-wide average (for unseen champions)
    """
    if pd.isna(champion) or champion in ("UNKNOWN", "", None):
        return global_avg_wr
    role_key = f"{champion}_{role}"
    if role_key in champ_role_wr_map:
        return champ_role_wr_map[role_key]
    if champion in champ_wr_global_map:
        return champ_wr_global_map[champion]
    return global_avg_wr


# ── 2.3 Synergy score for a team ─────────────────────────────────────────────
def team_synergy_score(champs, roles):
    """
    champs : list of champion names (may contain None for unpicked slots)
    roles  : matching list of role strings ("top", "jng", ...)
    Returns mean pair win rate for all valid pairs in the pair_map.
    Falls back to 0.5 if no pairs found.
    """
    role_champs = [
        (c, r) for c, r in zip(champs, roles)
        if c is not None and not (isinstance(c, float) and pd.isna(c)) and c != "UNKNOWN"
    ]
    if len(role_champs) < 2:
        return 0.5
    scores = []
    for (c1, r1), (c2, r2) in combinations(role_champs, 2):
        key = tuple(sorted([f"{c1}_{r1}", f"{c2}_{r2}"]))
        if key in pair_map:
            scores.append(pair_map[key])
    return float(np.mean(scores)) if scores else 0.5


# ── 2.4 Main feature engineering function ────────────────────────────────────
def build_features(df_pd, fit_encoders=True, encoders=None):
    """
    Takes a pandas DataFrame with at minimum the 10 role columns.
    Returns (df_features, encoders, feature_cols).

    fit_encoders=True  : training mode — fits new LabelEncoders
    fit_encoders=False : inference mode — uses existing encoders
    """
    df = df_pd.copy()

    # --- Layer 1: Role-specific win rate per slot ---
    for col in ROLE_COLS:
        role = COL_TO_ROLE[col]
        df[f"{col}_wr"] = df[col].apply(lambda c: get_wr(c, role))

    # --- Layer 2: Role-aware synergy score per team ---
    blue_role_names = [COL_TO_ROLE[c] for c in BLUE_ROLES]
    red_role_names  = [COL_TO_ROLE[c] for c in RED_ROLES]

    df["blue_synergy"] = df[BLUE_ROLES].apply(
        lambda row: team_synergy_score(row.tolist(), blue_role_names), axis=1
    )
    df["red_synergy"] = df[RED_ROLES].apply(
        lambda row: team_synergy_score(row.tolist(), red_role_names), axis=1
    )

    # --- Layer 3: Aggregate features & differentials ---
    df["blue_team_avg_wr"] = df[[f"{r}_wr" for r in BLUE_ROLES]].mean(axis=1)
    df["red_team_avg_wr"]  = df[[f"{r}_wr" for r in RED_ROLES]].mean(axis=1)
    df["wr_diff"]          = df["blue_team_avg_wr"] - df["red_team_avg_wr"]
    df["synergy_diff"]     = df["blue_synergy"]     - df["red_synergy"]

    # --- Layer 4: picks_filled — how complete is this draft? (0-10) ---
    df["picks_filled"] = df[ROLE_COLS].apply(
        lambda row: sum(1 for v in row if v not in (None, "UNKNOWN", "") and not (isinstance(v, float) and pd.isna(v))),
        axis=1
    )

    # --- Layer 5: Encode categoricals ---
    # Champion columns are encoded as champion_role strings (e.g. "Ashe_bot")
    # so the model sees role-specific identities directly.
    cat_cols = ROLE_COLS + ["patch", "league", "blue_team", "red_team"]

    if fit_encoders:
        encoders = {}

    for col in cat_cols:
        role = COL_TO_ROLE.get(col, None)
        if role:
            # For champion columns: encode as "champion_role"
            raw = df[col].apply(
                lambda c: f"{c}_{role}" if c not in (None, "") and not (isinstance(c, float) and pd.isna(c))
                          else f"UNKNOWN_{role}"
            )
        else:
            raw = df[col].fillna("UNKNOWN").astype(str)

        if fit_encoders:
            le = LabelEncoder()
            df[col + "_enc"] = le.fit_transform(raw)
            encoders[col] = le
        else:
            le = encoders[col]
            def safe_encode(val):
                if val in le.classes_:
                    return le.transform([val])[0]
                if "UNKNOWN" in le.classes_:
                    return le.transform(["UNKNOWN"])[0]
                unk_role = f"UNKNOWN_{role}" if role else "UNKNOWN"
                if unk_role in le.classes_:
                    return le.transform([unk_role])[0]
                return -1
            df[col + "_enc"] = raw.apply(safe_encode)

    # --- Assemble final feature list ---
    champ_enc_cols   = [c + "_enc" for c in ROLE_COLS]
    context_enc_cols = ["patch_enc", "league_enc"]
    numeric_cols     = (
        [f"{r}_wr" for r in ROLE_COLS] +
        ["blue_synergy", "red_synergy",
         "blue_team_avg_wr", "red_team_avg_wr",
         "wr_diff", "synergy_diff",
         "picks_filled"]
    )
    feature_cols = champ_enc_cols + context_enc_cols + numeric_cols

    return df, encoders, feature_cols


print("Feature engineering functions defined.")

In [ ]:
# ── 2.5 Apply feature engineering to full dataset ────────────────────────────
df = final_draft.to_pandas()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print(f"Date range: {df['date'].min()} → {df['date'].max()}")

df, encoders, feature_cols = build_features(df, fit_encoders=True)

print(f"\nTotal features: {len(feature_cols)}")
print(feature_cols)

---
## Section 3 — Model Training

In [ ]:
# ── 3.1 Temporal train / validation split ────────────────────────────────────
# 80% train (older matches), 20% val (newer matches)
# Temporal split avoids data leakage: model never sees future matches in training.
TARGET_COL = "blue_win"
cutoff = int(len(df) * 0.8)

train_df = df.iloc[:cutoff].copy()
val_df   = df.iloc[cutoff:].copy()

X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL]
X_val   = val_df[feature_cols]
y_val   = val_df[TARGET_COL]

print(f"Train: {len(train_df)} rows  ({train_df['date'].min().date()} → {train_df['date'].max().date()})")
print(f"Val  : {len(val_df)} rows  ({val_df['date'].min().date()} → {val_df['date'].max().date()})")
print(f"\nTrain target distribution:")
print(y_train.value_counts())

In [ ]:
# ── 3.2 Feature column references ────────────────────────────────────────────
# champ_enc_cols removed from feature_cols in build_features() so they are
# no longer in X_train. We only reference context_enc_cols here.
# No astype("category") marking — LabelEncoder integers are used as plain
# numerics, which avoids category-list mismatch errors at inference time.
champ_enc_cols   = [c + "_enc" for c in ROLE_COLS]
context_enc_cols = ["patch_enc", "league_enc"]
cat_enc_cols     = context_enc_cols   # only these remain in feature_cols

print("Feature column references set. No category dtype marking.")

In [ ]:
base_model = lgb.LGBMClassifier(
    objective="binary",
    metric="binary_logloss",
    n_estimators=600,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=30,
    min_data_per_group=20,
    cat_smooth=20,
    colsample_bytree=0.8,
    subsample=0.8,
    subsample_freq=1,
    reg_alpha=0.1,
    reg_lambda=0.5,
    verbose=-1,
    random_state=42
)

calibrated_model = CalibratedClassifierCV(
    base_model,
    method="isotonic",
    cv=5
)

# .astype(float) strips all pandas category dtypes before fitting,
# preventing "categorical_feature do not match" errors at inference time
calibrated_model.fit(X_train.astype(float), y_train)
print("Training complete.")

---
## Section 4 — Evaluation

In [ ]:
probs = calibrated_model.predict_proba(X_val.astype(float))[:, 1]

brier   = brier_score_loss(y_val, probs)
logloss = log_loss(y_val, probs)

print(f"Brier Score : {brier:.4f}   (random baseline = 0.2500)")
print(f"Log-Loss    : {logloss:.4f}   (random baseline = 0.6931)")
print(f"\nBrier improvement over random: {((0.25 - brier) / 0.25 * 100):.1f}%")

In [ ]:
fraction_pos, mean_pred = calibration_curve(y_val, probs, n_bins=10)

plt.figure(figsize=(6, 5))
plt.plot(mean_pred, fraction_pos, marker="o", label="Model")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
plt.xlabel("Predicted probability")
plt.ylabel("Actual win rate")
plt.title("Reliability diagram (calibration curve)")
plt.legend()
plt.tight_layout()
plt.savefig(f"{BASE}/calibration_curve_v2.png", dpi=150)
plt.show()
print("Saved: calibration_curve_v2.png")

In [ ]:
sample_row = val_df.iloc[0]

print(f"Sample game: {sample_row.get('gameid', 'N/A')}")
print(f"Blue team  : {sample_row.get('blue_team', 'N/A')}")
print(f"Red team   : {sample_row.get('red_team', 'N/A')}")
print(f"Actual result (blue_win): {sample_row['blue_win']}")
print()

DRAFT_ORDER = [
    "blue_top", "red_top", "red_jng", "blue_jng", "blue_mid",
    "red_mid",  "blue_bot", "red_bot", "red_sup",  "blue_sup"
]

partial_picks = {col: None for col in ROLE_COLS}
context = {
    "patch":     sample_row.get("patch",     "UNKNOWN"),
    "league":    sample_row.get("league",    "UNKNOWN"),
    "blue_team": sample_row.get("blue_team", "UNKNOWN"),
    "red_team":  sample_row.get("red_team",  "UNKNOWN")
}

print(f"{'#':>2}  {'Last Pick':<25}  {'Blue Win Prob':>13}")
print("-" * 46)

for i, col in enumerate(DRAFT_ORDER):
    partial_picks[col] = sample_row[col]
    row_dict = {**partial_picks, **context}
    temp_df = pd.DataFrame([row_dict])
    temp_df, _, _ = build_features(temp_df, fit_encoders=False, encoders=encoders)
    prob = calibrated_model.predict_proba(temp_df[feature_cols].astype(float))[:, 1][0]
    print(f"  {i+1:>2}  {col:<10} = {str(sample_row[col]):<14}  {prob:.1%}")

---
## Section 5 — SHAP Analysis

In [ ]:
# ── 5.1 Extract raw LightGBM model from calibrator ───────────────────────────
# CalibratedClassifierCV wraps multiple fold models; use fold 0 for SHAP.
lgbm_model = calibrated_model.calibrated_classifiers_[0].estimator
X_test = val_df[feature_cols]

print("Initializing TreeExplainer...")
explainer = shap.TreeExplainer(lgbm_model)

print("Computing SHAP values (using .values to avoid metadata issues)...")
shap_values = explainer(X_test.values)
shap_values.feature_names = feature_cols

print(f"SHAP values shape: {shap_values.shape}")

In [ ]:
# ── 5.2 Global SHAP plots ────────────────────────────────────────────────────
# Beeswarm: shows which features drive predictions and in what direction
shap.plots.beeswarm(shap_values, show=False)
plt.savefig(f"{BASE}/beeswarm_v2.png", bbox_inches="tight", dpi=150)
plt.clf()
print("Saved: beeswarm_v2.png")

# Bar: mean absolute SHAP value per feature (overall importance)
shap.plots.bar(shap_values, show=False)
plt.savefig(f"{BASE}/bar_v2.png", bbox_inches="tight", dpi=150)
plt.clf()
print("Saved: bar_v2.png")

In [ ]:
def explain_single_draft(match_index=0, save_plot=True):
    print(f"--- Explaining match index {match_index} ---")

    if save_plot:
        shap.plots.waterfall(shap_values[match_index], show=False)
        plt.savefig(f"{BASE}/waterfall_match_{match_index}.png", bbox_inches="tight", dpi=150)
        plt.clf()
        print(f"Saved: waterfall_match_{match_index}.png")

    single_row = X_test.iloc[match_index]
    local_shap = shap_values[match_index]

    impacts = []
    for col, val, sv in zip(feature_cols, single_row, local_shap.values):
        if abs(sv) > 0.01:
            impacts.append({
                "feature":            col,
                "value":              float(val),
                "impact_on_win_prob": float(sv)
            })

    impacts.sort(key=lambda x: abs(x["impact_on_win_prob"]), reverse=True)

    payload = {
        "match_index":        match_index,
        "base_win_prob":      float(local_shap.base_values),
        "predicted_win_prob": float(calibrated_model.predict_proba(
                                  X_val.astype(float).iloc[[match_index]]
                              )[:, 1][0]),
        "top_drivers":        impacts[:5]
    }
    return payload


payload = explain_single_draft(0)
print("\nAgent payload:")
print(json.dumps(payload, indent=2))

---
## Section 6 — Save Artifacts

In [ ]:
# ── 6.1 Save all model artifacts ─────────────────────────────────────────────
ARTIFACTS = [
    (calibrated_model,   "calibrated_model_v2.pkl"),
    (feature_cols,       "feature_cols_v2.pkl"),
    (encoders,           "encoders_v2.pkl"),
    (champ_role_wr_map,  "champ_role_wr_map.pkl"),
    (champ_wr_global_map,"champ_wr_global_map.pkl"),
    (pair_map,           "pair_map.pkl"),
    (global_avg_wr,      "global_avg_wr.pkl"),
]

for obj, filename in ARTIFACTS:
    path = f"{BASE}/{filename}"
    with open(path, "wb") as fh:
        pickle.dump(obj, fh)
    print(f"Saved: {filename}")

# Also save SHAP output for the frontend agent
shap_output = {
    "values":   shap_values.values,
    "features": feature_cols,
    "data":     X_test.values
}
with open(f"{BASE}/shap_output_v2.pkl", "wb") as fh:
    pickle.dump(shap_output, fh)
print("Saved: shap_output_v2.pkl")

print("\nAll artifacts saved.")

---
## Section 7 — Inference Function (for the Frontend)

Copy this section into your backend `inference.py`. Load artifacts once at startup, then call `predict_draft()` after every champion pick.

In [ ]:
def load_artifacts(base_path):
    with open(f"{base_path}/calibrated_model_v2.pkl", "rb") as fh:
        model = pickle.load(fh)
    with open(f"{base_path}/feature_cols_v2.pkl", "rb") as fh:
        feat_cols = pickle.load(fh)
    with open(f"{base_path}/encoders_v2.pkl", "rb") as fh:
        enc = pickle.load(fh)
    with open(f"{base_path}/champ_role_wr_map.pkl", "rb") as fh:
        cr_map = pickle.load(fh)
    with open(f"{base_path}/champ_wr_global_map.pkl", "rb") as fh:
        cg_map = pickle.load(fh)
    with open(f"{base_path}/pair_map.pkl", "rb") as fh:
        pm = pickle.load(fh)
    with open(f"{base_path}/global_avg_wr.pkl", "rb") as fh:
        g_avg = pickle.load(fh)
    return model, feat_cols, enc, cr_map, cg_map, pm, g_avg


def predict_draft(picks, patch="UNKNOWN", league="UNKNOWN",
                  blue_team="UNKNOWN", red_team="UNKNOWN"):
    full_picks = {col: picks.get(col, None) for col in ROLE_COLS}
    row = {**full_picks, "patch": patch, "league": league,
           "blue_team": blue_team, "red_team": red_team}

    temp_df = pd.DataFrame([row])
    temp_df, _, _ = build_features(temp_df, fit_encoders=False, encoders=encoders)

    # .astype(float) prevents "categorical_feature do not match" at inference
    prob = float(calibrated_model.predict_proba(
        temp_df[feature_cols].astype(float)
    )[:, 1][0])

    # Clamp raw output: pro drafts realistically stay within 25–75%
    prob = max(0.25, min(0.75, prob))

    # Dampen toward 50% for partial drafts
    filled = int(temp_df["picks_filled"].iloc[0])
    confidence_weight = filled / 10.0
    prob = 0.5 + (prob - 0.5) * confidence_weight

    if filled <= 3:
        confidence = "low"
    elif filled <= 7:
        confidence = "medium"
    else:
        confidence = "high"

    return {
        "blue_win_prob": round(prob, 4),
        "picks_filled":  filled,
        "confidence":    confidence
    }


DEMO_JINX = [
    ("blue_top", "Garen"),   ("red_top",  "Darius"),
    ("red_jng",  "Vi"),      ("blue_jng", "LeeSin"),
    ("blue_mid", "Azir"),    ("red_mid",  "Orianna"),
    ("blue_bot", "Jinx"),    ("red_bot",  "Caitlyn"),
    ("red_sup",  "Thresh"),  ("blue_sup", "Nautilus"),
]

DEMO_META = [
    ("blue_top", "Aatrox"),    ("red_top",  "Gnar"),
    ("red_jng",  "Graves"),    ("blue_jng", "Viego"),
    ("blue_mid", "Viktor"),    ("red_mid",  "Orianna"),
    ("blue_bot", "Aphelios"),  ("red_bot",  "Kaisa"),
    ("red_sup",  "Lulu"),      ("blue_sup", "Leona"),
]

def run_demo(draft, label):
    current_picks = {col: None for col in ROLE_COLS}
    print(f"\n{'='*58}")
    print(f"  {label}")
    print(f"{'='*58}")
    print(f"{'#':>2}  {'Slot':<10} {'Champion':<12} {'Blue Win%':>9}  {'Picks':>5}  {'Confidence'}")
    print("-" * 58)
    for i, (slot, champ) in enumerate(draft):
        current_picks[slot] = champ
        result = predict_draft(current_picks)
        print(f"{i+1:>2}  {slot:<10} {champ:<12} {result['blue_win_prob']:>8.1%}  "
              f"{result['picks_filled']:>5}  {result['confidence']}")
    print(f"\n  Final blue win probability: {result['blue_win_prob']:.1%}")

run_demo(DEMO_JINX, "Draft A — Jinx composition")
run_demo(DEMO_META, "Draft B — Strong meta draft")